# Why do Alzheimer's trials stop?

In [1]:
import pandas as pd
import sys
sys.path.append("..")
from src.fixed_data import fix_data
df = pd.read_csv("../data/processed/alzheimer.csv")
df = fix_data(df)

In [2]:
from src.analysis import categorize_text

I have taken 3 types of trials because trials being stopped does not mean only terminated but withdrawn and suspended as well

In [3]:
mask = df["status"].isin(["TERMINATED","SUSPENDED","WITHDRAWN"])
stopped = df[mask].copy()

In [4]:
stopped.shape

(413, 11)

The amount of stopped trials has changed from 304 to 413 as we have increased the amount of stopped trials with the addition of suspended and withdrawn.

In [5]:
stopped["status"].value_counts()
stopped.head(20)

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results
12,NCT05310071,A Study to Verify the Clinical Benefit of Aduc...,TERMINATED,PHASE3,INTERVENTIONAL,2022-06-02,Sponsor's Decision,1027,2024-08-12,INDUSTRY,True
15,NCT03980730,Study of Azeliragon in Patients With Mild Alzh...,TERMINATED,PHASE2,INTERVENTIONAL,2019-06-27,for business reasons,43,2021-01-29,INDUSTRY,True
18,NCT01979432,The Cognitive and Cardiovascular Health in the...,TERMINATED,NaN,INTERVENTIONAL,2013-12-15,Lack of patients,17,2015-12-12,OTHER,False
35,NCT01560585,Open Label Study of Isotretinoin in Mild to Mo...,TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2012-04-15,Adverse events in 2/3 participants; study was ...,3,2014-08-15,OTHER,True
37,NCT02263235,In Vivo Alzheimer Proteomics,TERMINATED,NaN,INTERVENTIONAL,2013-10-08,end of the inclusion period,89,2018-05-22,OTHER,False
40,NCT00916617,Study Evaluating The Long-Term Safety And Effi...,TERMINATED,PHASE2,INTERVENTIONAL,2009-06-15,"The study was terminated on August 6, 2012, be...",62,2012-09-15,INDUSTRY,True
47,NCT05402423,An Integrated Intervention of Computerized Cog...,WITHDRAWN,NaN,INTERVENTIONAL,2022-06-27,It was not possible to acquire the medical device,0,2023-06-26,OTHER,False
48,NCT03484143,Neuro RX Gamma - Pivotal Phase,SUSPENDED,NaN,INTERVENTIONAL,2019-06-26,Recruitment has been too slow. We are consider...,228,2023-05-15,INDUSTRY,False
54,NCT02791191,A Study of LY3202626 on Disease Progression in...,TERMINATED,PHASE2,INTERVENTIONAL,2016-06-16,Low likelihood of identifying a statistically ...,316,2018-07-02,INDUSTRY,True
56,NCT02781220,Implications for Management of PET Amyloid Cla...,TERMINATED,NaN,OBSERVATIONAL,2016-07-15,IRB Study Closure - No HRPP Progress Update,69,2023-06-10,OTHER,False


Going through 20 studies, looking at why they were stopped(why_stopped) made me realize that almost every reason is written differently even when its the same idea. So I cannot count them as they are.

## Cleaning the data

In [6]:
stopped["cleaned_data"] =stopped["why_stopped"].str.lower().str.strip()

Checking how many reasons mention fund before building categories

In [7]:
(stopped["cleaned_data"].str.contains("fund",na=False)).sum()


np.int64(31)

## Categories
Each reason is checked in this order and gets the first category it matches:

1. **missing** -> no reason given
2. **efficacy** -> efficacy, futility
3. **safety** -> safety, adverse, death
4. **investigator** -> pi, investigator
5. **covid** -> covid
6. **administrative** -> administrative
7. **recruitment** -> enrol, recruit, accrual, particip
8. **funding** -> sponsor, business, fund, strateg, focus
9. **see description** -> description (the reason is only in the detailed description)
10. **other** -> matches nothing

The order matters because some reasons match more than one category. For example "sponsor decision due to safety concerns" has both sponsor and safety. Specific reasons like efficacy and safety are checked first and vague ones like sponsor last, otherwise sponsor would steal them and funding would look bigger than it really is.

## Improving the categories
My first version only had funding, recruitment, efficacy and safety, and 44% of the reasons ended up as "other". To fix that I looked at the most common words in the "other" reasons, which showed groups I was missing like the PI leaving, COVID and administrative reasons. I added those as categories, removed keywords that were too broad (like "left", "patient" and "time") and checked 15 random trials to see if the categories made sense. In the end "other" went down to 29%.

In [8]:
stopped["reason_category"] = stopped["cleaned_data"].apply(categorize_text)

In [9]:
(stopped["reason_category"].value_counts(normalize= True)*100).round(2)

reason_category
other              28.57
recruitment        19.13
missing            14.04
funding            11.38
efficacy            9.44
covid               6.30
investigator        5.08
safety              2.66
administrative      1.94
see description     1.45
Name: proportion, dtype: float64

This shows that most Alzheimer’s trials do not stop because the drug failed (efficacy + safety ≈ 12%) but because of practical problems like not finding patients, running out of money or COVID (≈44%).

In [10]:
(pd.crosstab(stopped["phases"],stopped["reason_category"],normalize ="index" )*100).round(2)

reason_category,administrative,covid,efficacy,funding,investigator,missing,other,recruitment,safety,see description
phases,,,,,,,,,,
EARLY_PHASE1,0.00,11.11,0.00,11.11,0.00,0.00,55.56,22.22,0.00,0.00
PHASE1,0.00,5.66,11.32,9.43,3.77,15.09,35.85,16.98,1.89,0.00
"PHASE1, PHASE2",12.50,12.50,0.00,12.50,0.00,0.00,37.50,12.50,12.50,0.00
PHASE2,2.27,2.27,12.50,12.50,1.14,23.86,26.14,14.77,3.41,1.14
"PHASE2, PHASE3",0.00,0.00,27.27,0.00,0.00,27.27,27.27,9.09,9.09,0.00
PHASE3,1.49,1.49,23.88,11.94,2.99,8.96,26.87,8.96,7.46,5.97
PHASE4,7.69,0.00,0.00,7.69,0.00,23.08,15.38,46.15,0.00,0.00


In [11]:
(pd.crosstab(stopped["phases"],stopped["reason_category"]))

reason_category,administrative,covid,efficacy,funding,investigator,missing,other,recruitment,safety,see description
phases,,,,,,,,,,
EARLY_PHASE1,0,1,0,1,0,0,5,2,0,0
PHASE1,0,3,6,5,2,8,19,9,1,0
"PHASE1, PHASE2",1,1,0,1,0,0,3,1,1,0
PHASE2,2,2,11,11,1,21,23,13,3,1
"PHASE2, PHASE3",0,0,3,0,0,3,3,1,1,0
PHASE3,1,1,16,8,2,6,18,6,5,4
PHASE4,1,0,0,1,0,3,2,6,0,0


 ## Reasons by phase
**Phase 3:** efficacy is the most common named reason (16 trials, 24%), so the drug not working shows up most in the big late trials. This fits with Phase 3 having the highest termination rate (20%).

**Phase 2:** recruitment is the most common named reason (13 trials, 15%), just ahead of efficacy and funding (11 trials each). But 24% of stopped Phase 2 trials give no reason, so the picture is less clear.

**Caveat:** some phases have very few stopped trials (Phase 4: 13, Phase 2/3: 11, Early Phase 1: 9, Phase 1/2: 8), so their % can change a lot with one trial. For example Phase 4 shows 46% recruitment, but that's only 6 trials.

## Limitations
- The reasons were sorted with keyword matching, so 29% still end up as "other" because they don't contain any of the keywords.
- Reasons written in an indirect way get missed. For example "scans showed no difference" means the drug didn't work, but it doesn't contain "efficacy", so it lands in "other". This means efficacy and safety are probably undercounted.
- 14% of stopped trials give no reason at all, so we can't know why they stopped.
- 6 trials only say the reason is in the detailed description, which wasn't included in this analysis.